# NB09 — Build04 Regulatory Investigation Agent (MVP v1)

**Scientific AI Builder · Frozen EU regulatory benchmark v1 · 2026-10-20**

**Goal:** Extend the NB08 Build03-B frozen legal retriever and Cellar metadata client with a **LangGraph investigation workflow:**

`Planner → Retriever → Evidence Validator → (Targeted Re-search if needed) → Report → Final Claim Audit`.

**Expected improvements:** reduce the Build03 Q4 cross-document amendment error, Q5/Q7/Q9 missing evidence, and Q10 incomplete claim-to-provision mapping. These are hypotheses to test, **not proven improvements**.

### Experiment protections
- Same five-law frozen snapshots, **1,383 legal-aware chunks**, MiniLM + BM25 + RRF and ordered Top-5 parity **50/50** must pass.
- Same `gpt-6-luna` model and `medium` reasoning by default; added workflow/LLM calls are a declared treatment (not an isolated causal comparison).
- **No verified answers, Q-specific Must-have lists or previously assigned grades** are supplied to the Planner/Validator/Reporter. The benchmark questions are the only evaluation inputs.
- Cellar **consolidation metadata ≠ article-level current law**. The notebook must never silently upgrade a metadata lookup to proof of legal applicability.
- **All billable OpenAI calls are disabled by default**. Use the offline gates first, then explicitly enable a single smoke-test question.
- The output is a regulatory research prototype, not shipment clearance or production legal advice.

**Run from the root of `hyejeong0617/Regulatory_agent` with the original `data/` directory.** If using Colab, set the working directory accordingly. The new `NB09...ipynb` belongs beside NB08/NB07; no earlier files are modified.

## 0. Installation and configuration

If packages are missing, uncomment the install line and restart the kernel. Model downloads for SentenceTransformers, OpenAI requests, and Cellar requests require network access.

In [1]:
%pip install -U "langgraph>=0.2" "pydantic>=2" "openai>=1.60" 
#"sentence-transformers" "rank-bm25" numpy pandas requests beautifulsoup4


   ---------------------------------------- 0.0/573.8 kB ? eta -:--:--
   ---------------------------------------- 573.8/573.8 kB 9.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/869.0 kB ? eta -:--:--
   ---------------------------------------- 869.0/869.0 kB 5.9 MB/s eta 0:00:00

  Attempting uninstall: orjson

    Found existing installation: orjson 3.11.4

    Uninstalling orjson-3.11.4:

      Successfully uninstalled orjson-3.11.4

   ---- ----------------------------------- 1/9 [orjson]
   ---- ----------------------------------- 1/9 [orjson]
   ---- ----------------------------------- 1/9 [orjson]
   ---- ----------------------------------- 1/9 [orjson]
   ---- ----------------------------------- 1/9 [orjson]
   ---- ----------------------------------- 1/9 [orjson]
   ---- ----------------------------------- 1/9 [orjson]
   ---- ----------------------------------- 1/9 [orjson]
   ---- ----------------------------------- 1/9 [orjson]
   ---- ----------------

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 1.0.8 requires langgraph<1.1.0,>=1.0.2, but you have langgraph 1.2.16 which is incompatible.
langchain-huggingface 1.0.1 requires huggingface-hub<1.0.0,>=0.33.4, but you have huggingface-hub 1.33.0 which is incompatible.
langchain-openai 1.0.3 requires openai<3.0.0,>=1.109.1, but you have openai 3.35.1 which is incompatible.


In [2]:
import os, re, json, hashlib, platform, time, math
from datetime import datetime, timezone, date
from pathlib import Path
from getpass import getpass
from typing import TypedDict, Literal, NotRequired
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import requests
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from pydantic import BaseModel, Field, ValidationError
from langgraph.graph import StateGraph, START, END
from openai import OpenAI

DATA_DIR = Path("data")
assert DATA_DIR.is_dir(), "Run NB09 from Regulatory_agent repository root (data/ missing)."
MODEL_NAME = "gpt-6-luna"
REASONING_EFFORT = "medium"
RUN_OFFICIAL = False             # Set True deliberately ONLY for paid calls.
SELECT_QIDS = ["Q4"]             # First controlled smoke test; change later.
ENABLE_CELLAR_LIVE = False       # Optional network metadata tool; NOT article verification.
MAX_INITIAL_SEARCHES = 5
MAX_TARGETED_SEARCHES = 3
MAX_TOTAL_SEARCHES = 8
MAX_VALIDATION_ROUNDS = 2
MAX_CELLAR_CALLS = 2
MAX_LLM_CALLS = 12              # Cost stop, including planner/validator/report/audit.
MAX_OUTPUT_TOKENS = 2200         # Each model call, not total budget.
EXPERIMENT_ID = "NB09_build04_investigation_v1"
RUN_DIR = DATA_DIR / "build04_nb09_runs"
print("Python:", platform.python_version(), "| model:", MODEL_NAME)
print("Paid calls enabled:", RUN_OFFICIAL, "| live Cellar:", ENABLE_CELLAR_LIVE)

c:\Users\SAMSUNG\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Python: 3.13.5 | model: gpt-6-luna
Paid calls enabled: False | live Cellar: False


## 1. Load unchanged NB08/NB04b assets (frozen data gate)

The notebook reads the exact NB08 library and retrieval configuration; it does **not** regenerate or silently update them. The NB08 `benchmark_context` contains shared scenario information—not the evaluator's Verified Reference.

In [4]:
FILES = {
    "chunks": DATA_DIR / "chunks_legal_aware.json",
    "benchmark": DATA_DIR / "benchmark_questions_v1.json",
    "retrieval_config": DATA_DIR / "retrieval_config_nb03_final_hybrid_static_rag.json",
    "nb03_results": DATA_DIR / "retrieval_results_nb03_final_hybrid_static_rag.csv",
    "nb04b_config": DATA_DIR / "nb04b_context_repaired_experiment_config.json",
    "nb04b_answers": DATA_DIR / "nb04b_context_repaired_openai_luna_answers_comparison.csv",
}
for key, path in FILES.items():
    assert path.is_file(), f"Missing {key}: {path}"

legal_chunks = json.loads(FILES["chunks"].read_text(encoding="utf-8"))
benchmark = json.loads(FILES["benchmark"].read_text(encoding="utf-8"))
retrieval_config = json.loads(FILES["retrieval_config"].read_text(encoding="utf-8"))
nb04b_config = json.loads(FILES["nb04b_config"].read_text(encoding="utf-8"))
nb03_df = pd.read_csv(FILES["nb03_results"], encoding="utf-8-sig")
comparison_df = pd.read_csv(FILES["nb04b_answers"], encoding="utf-8-sig")
questions = benchmark["questions"]
benchmark_hash = hashlib.sha256(json.dumps(questions, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
assert len(legal_chunks) == 1383, "Library changed: investigate before running controlled test"
assert list(questions) == [f"Q{i}" for i in range(1,11)]
assert benchmark_hash == retrieval_config["benchmark_sha256"]
assert nb04b_config["model"] == MODEL_NAME
assert nb04b_config["reasoning_effort"] == REASONING_EFFORT
assert retrieval_config["retrieval_method"] == "hybrid_dense_bm25_rrf_diverse_v1"
assert retrieval_config["top_k"] == 5
assert retrieval_config["rrf_k"] == 60
assert retrieval_config["candidate_depth_per_retriever"] == 50
assert retrieval_config["diversification"]["max_per_group"] == 2
assert set(comparison_df.qid) == set(questions)

print("Benchmark:", benchmark["benchmark_id"])
print("Legal chunks:", len(legal_chunks))
print("Benchmark questions:", len(questions))
print("Frozen NB04b answers available:", comparison_df.qid.nunique())

Benchmark: regulatory_agent_v1_notion_2026-10-08
Legal chunks: 1383
Benchmark questions: 10
Frozen NB04b answers available: 10


## 2. Reuse NB08's exact hybrid search and ordered parity gate

Embedding computation can take time on the first run. The **50/50 gate must pass** before any official Build04 evaluation. During official runs, `hybrid_search` uses the identical five-law corpus and retrieval mechanics as NB03/NB08.

In [5]:
def safe(value):
    return "" if value is None else str(value)

def searchable_text(chunk):
    parts = [safe(chunk.get(k)) for k in (
        "regulation_title", "file", "location", "article", "heading", "text"
    )]
    return " | ".join(p for p in parts if p).strip()

TOKEN_RE = re.compile(r"[A-Za-zÀ-ÿ0-9]+(?:[.,][0-9]+)?%?")
def tokenize(text):
    return TOKEN_RE.findall(text.casefold())

legal_texts = [searchable_text(ch) for ch in legal_chunks]
assert all(legal_texts)
EMBEDDING_MODEL = retrieval_config["embedding_model"]
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
legal_embeddings = embedding_model.encode(
    legal_texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True
)
bm25 = BM25Okapi([tokenize(t) for t in legal_texts])
print("NB03-equivalent retriever components ready:", EMBEDDING_MODEL)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

NB03-equivalent retriever components ready: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [6]:
RRF_K = int(retrieval_config["rrf_k"])
CANDIDATE_DEPTH = int(retrieval_config["candidate_depth_per_retriever"])
MAX_PER_SOURCE_GROUP = int(retrieval_config["diversification"]["max_per_group"])

def source_group(chunk):
    file_name = safe(chunk.get("file"))
    article = safe(chunk.get("article"))
    location = safe(chunk.get("location"))
    return (file_name, article if article else location)

def hybrid_search(query: str, top_k: int = 5):
    query = str(query).strip()
    if not query:
        raise ValueError("query must be non-empty")
    if not (1 <= int(top_k) <= 10):
        raise ValueError("top_k must be in 1..10")
    q_emb = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False
    )[0]
    dense_scores = legal_embeddings @ q_emb
    bm25_scores = np.asarray(bm25.get_scores(tokenize(query)), dtype=float)
    dense_order = np.argsort(dense_scores)[::-1][:CANDIDATE_DEPTH]
    bm25_order = np.argsort(bm25_scores)[::-1][:CANDIDATE_DEPTH]
    fused_scores = {}
    for rank, idx in enumerate(dense_order, start=1):
        idx = int(idx)
        fused_scores[idx] = fused_scores.get(idx, 0.) + 1. / (RRF_K + rank)
    for rank, idx in enumerate(bm25_order, start=1):
        idx = int(idx)
        fused_scores[idx] = fused_scores.get(idx, 0.) + 1. / (RRF_K + rank)
    fused_order = sorted(fused_scores, key=fused_scores.get, reverse=True)
    selected, counts = [], {}
    for idx in fused_order:
        chunk = legal_chunks[idx]
        group = source_group(chunk)
        if counts.get(group, 0) >= MAX_PER_SOURCE_GROUP:
            continue
        selected.append((idx, chunk, fused_scores[idx]))
        counts[group] = counts.get(group, 0) + 1
        if len(selected) == int(top_k):
            break
    results = []
    for rank, (idx, chunk, rrf_score) in enumerate(selected, start=1):
        results.append({
            "rank": rank, "chunk_index": int(idx),
            "chunk_id": chunk.get("chunk_id"), "rrf_score": float(rrf_score),
            "file": chunk.get("file"), "regulation_title": chunk.get("regulation_title"),
            "article": chunk.get("article"), "location": chunk.get("location"),
            "heading": chunk.get("heading"), "text": chunk.get("text"),
        })
    return results

In [7]:
PARITY_PATH = DATA_DIR / "build04_nb09_retrieval_parity_audit.csv"
audit_rows, parity_by_qid = [], {}
for qid, question in questions.items():
    baseline = nb03_df[nb03_df.qid == qid].sort_values("rank")
    expected = baseline.chunk_id.astype(str).tolist()
    actual = [r["chunk_id"] for r in hybrid_search(question, top_k=5)]
    assert len(expected) == len(actual) == 5
    parity_by_qid[qid] = expected == actual
    for rank, (a,b) in enumerate(zip(expected,actual), start=1):
        audit_rows.append({"qid":qid,"rank":rank,"nb03_chunk_id":a,"nb09_chunk_id":b,"match":a==b})
    print(f"{qid}: {'PASS' if parity_by_qid[qid] else 'MISMATCH'}")
audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(PARITY_PATH, index=False, encoding="utf-8-sig")
matched = int(audit_df.match.sum())
PARITY_OK = matched == 50 and all(parity_by_qid.values())
print(f"\nPARITY: {matched}/50 ordered chunks | Saved: {PARITY_PATH}")
assert PARITY_OK, "STOP: original NB03 retrieval parity failed. No official NB09 run allowed."

Q1: PASS
Q2: PASS
Q3: PASS
Q4: PASS
Q5: PASS
Q6: PASS
Q7: PASS
Q8: PASS
Q9: PASS
Q10: PASS

PARITY: 50/50 ordered chunks | Saved: data\build04_nb09_retrieval_parity_audit.csv


## 3. Reuse NB08 Cellar metadata checker unchanged

This is the **existing** `retrieve_cellar_current_law` implementation, including CELEX allowlisting, official EU Publications Office SPARQL, failure-closed parsing and flags `article_level_changes_verified=False`, `current_legal_status_verified=False`. The returned date is only a **latest-observed consolidation candidate**. The web request is **not made** until an enabled workflow asks for it.

In [8]:
LAW_REGISTRY = {
    "2023/915": {"celex":"32023R0915", "eli":"reg/2023/915", "snapshot_date":"2026-08-19", "snapshot_file":"EU_2023_915_consolidated_2026-08-19.html", "title":"Contaminants maximum levels"},
    "2019/1793": {"celex":"32019R1793", "eli":"reg_impl/2019/1793", "snapshot_date":"2026-06-30", "snapshot_file":"EU_2019_1793_consolidated_2026-06-30.html", "title":"Increased import controls / special conditions"},
    "2017/625": {"celex":"32017R0625", "eli":"reg/2017/625", "snapshot_date":"2025-01-05", "snapshot_file":"EU_2017_625_consolidated_2025-01-05.html", "title":"Official controls"},
    "178/2002": {"celex":"32002R0178", "eli":"reg/2002/178", "snapshot_date":"2026-01-01", "snapshot_file":"EU_2002_178_consolidated_2026-01-01.html", "title":"General Food Law"},
    "2019/1013": {"celex":"32019R1013", "eli":"reg_impl/2019/1013", "snapshot_date":None, "snapshot_file":"EU_2019__1013_original.html", "title":"Prior-notification arrangements"},
}

for key, item in LAW_REGISTRY.items():
    assert (DATA_DIR/item["snapshot_file"]).is_file(), f"Snapshot missing for {key}"
    item["official_basic_url"] = f'https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX%3A{item["celex"]}'

assert set(LAW_REGISTRY) == {"2023/915","2019/1793","2017/625","178/2002","2019/1013"}
print("Current-law checker scope:", list(LAW_REGISTRY))

Current-law checker scope: ['2023/915', '2019/1793', '2017/625', '178/2002', '2019/1013']


In [9]:
# Official Cellar SPARQL: CELEX identifiers are stored under
# cdm:resource_legal_id_celex (NOT cdm:work_id_document).
# No EUR-Lex HTML scraping, no user-supplied URLs, no fabricated latest dates.
from urllib.parse import urlparse

CELLAR_SPARQL = "https://publications.europa.eu/webapi/rdf/sparql"
CELLAR_CELEX_BASE = "https://publications.europa.eu/resource/celex/"
CELLAR_TIMEOUT = (10, 45)
MAX_SPARQL_BYTES = 2_000_000
MAX_VERSIONS = 100


def _base_record(regulation):
    law = LAW_REGISTRY.get(regulation, {})
    return {
        "regulation": regulation, "basic_celex": law.get("celex"),
        "snapshot_date": law.get("snapshot_date"),
        "snapshot_file": law.get("snapshot_file"),
        "official_basic_url": law.get("official_basic_url"),
        "official_metadata_endpoint": CELLAR_SPARQL,
        "checked_utc": datetime.now(timezone.utc).isoformat(),
        "status": "unverified", "source": "EU Publications Office / Cellar SPARQL",
        "latest_date_candidate": None, "latest_celex_candidate": None,
        "observed_consolidations": [], "date_comparison": "not_verified",
        "article_level_changes_verified": False,
        "application_date_verified": False,
        "current_legal_status_verified": False,
        "complete_consolidation_list_verified": False,
    }


def _version_celex_pattern(basic_celex):
    # Sector 3 basic act -> sector 0 consolidated versions.
    return re.compile(r"^0" + re.escape(basic_celex[1:]) + r"-(\d{8})$")


def _sparql_versions_query(celex, mode="relation"):
    assert re.fullmatch(r"3\d{4}R\d{4}", celex), "Only allowlisted regulation CELEX IDs"
    prefix = "0" + celex[1:] + "-"
    header = "PREFIX cdm: <http://publications.europa.eu/ontology/cdm#>\n"
    if mode == "relation":
        # Both documented relation predicates; incoming edges point at base work.
        return header + f"""SELECT DISTINCT ?consolidated ?consolidated_celex WHERE {{
  ?original cdm:resource_legal_id_celex "{celex}" .
  {{ ?consolidated cdm:act_consolidated_based_on_resource_legal ?original . }}
  UNION
  {{ ?consolidated cdm:act_consolidated_consolidates_resource_legal ?original . }}
  ?consolidated cdm:resource_legal_id_celex ?consolidated_celex .
  FILTER(STRSTARTS(STR(?consolidated_celex), "{prefix}"))
}} ORDER BY DESC(?consolidated_celex) LIMIT {MAX_VERSIONS}"""
    if mode == "celex_index":
        # Validated against live Cellar: the plain SPARQL CONTAINS filter
        # returned 19 CELEX records (including 02019R1793-20260630).
        # The old Virtuoso bif:contains clause silently returned zero rows.
        # Search broadly, then use _parse_versions() to accept ONLY the
        # correct sector-0 consolidated CELEX IDs with valid dates.
        search_fragment = celex[1:]  # e.g. 2019R1793
        return header + f"""SELECT DISTINCT ?consolidated ?consolidated_celex WHERE {{
  ?consolidated cdm:resource_legal_id_celex ?consolidated_celex .
  FILTER(CONTAINS(STR(?consolidated_celex), "{search_fragment}"))
}} ORDER BY DESC(?consolidated_celex) LIMIT {MAX_VERSIONS}"""
    raise ValueError("Unknown SPARQL query mode")


def _run_cellar_sparql(query, session=None):
    client = session if session is not None else requests.Session()
    response = client.get(
        CELLAR_SPARQL, params={"query": query},
        headers={"Accept": "application/sparql-results+json",
                 "User-Agent": "ScientificAIBuilder-Research/0.2"},
        timeout=CELLAR_TIMEOUT, allow_redirects=True,
    )
    resolved = getattr(response, "url", CELLAR_SPARQL)
    parsed = urlparse(resolved)
    if parsed.scheme != "https" or parsed.hostname != "publications.europa.eu":
        raise ValueError("untrusted_redirect")
    if response.status_code != 200:
        raise ValueError(f"http_{response.status_code}")
    raw = response.content
    if not raw or len(raw) > MAX_SPARQL_BYTES:
        raise ValueError("empty_or_oversize_response")
    # Virtuoso can return HTTP 200 with plain-text query error.
    content_type = response.headers.get("Content-Type", "").lower()
    if "json" not in content_type:
        raise ValueError("non_json_or_sparql_error")
    data = response.json()
    if not isinstance(data.get("results", {}).get("bindings"), list):
        raise ValueError("invalid_sparql_result_structure")
    return data["results"]["bindings"]


def _parse_versions(bindings, basic_celex, as_of=None):
    as_of = as_of or datetime.now(timezone.utc).date()
    pattern = _version_celex_pattern(basic_celex)
    found = {}
    for row in bindings:
        value = row.get("consolidated_celex", {}).get("value", "")
        match = pattern.fullmatch(value)
        if not match:
            continue
        try:
            dt = datetime.strptime(match.group(1), "%Y%m%d").date()
        except ValueError:
            continue
        # Future-dated consolidations are recorded separately, not current.
        if dt > as_of:
            continue
        found[value] = {
            "celex": value,
            "date": dt.isoformat(),
            "official_celex_url": CELLAR_CELEX_BASE + value,
        }
    return sorted(found.values(), key=lambda v: (v["date"],v["celex"]), reverse=True)


def retrieve_cellar_current_law(regulation, session=None, as_of=None):
    """Official metadata check only; NEVER certify legal force or amended clauses.

    Relational lookup + independently scoped CELEX-index lookup.
    Missing results or failed confirmation are UNVERIFIED, not 'no update'.
    """
    rec = _base_record(regulation)
    if regulation not in LAW_REGISTRY:
        rec["error"] = "not_allowlisted"
        return rec
    basic = LAW_REGISTRY[regulation]["celex"]
    as_of = as_of or datetime.now(timezone.utc).date()
    if isinstance(as_of, str):
        as_of = date.fromisoformat(as_of)
    rec["as_of_date"] = as_of.isoformat()

    try:
        rel = _run_cellar_sparql(_sparql_versions_query(basic,"relation"),session)
        rec["relation_row_count"] = len(rel)
    except (requests.RequestException, ValueError, KeyError, TypeError) as exc:
        rel = []
        rec["relation_error"] = f"{type(exc).__name__}: {exc}"

    # Important: index query is independent, not assumed equivalent to links.
    try:
        idx = _run_cellar_sparql(_sparql_versions_query(basic,"celex_index"),session)
        rec["celex_index_row_count"] = len(idx)
    except (requests.RequestException, ValueError, KeyError, TypeError) as exc:
        idx = []
        rec["celex_index_error"] = f"{type(exc).__name__}: {exc}"

    observed = _parse_versions(rel+idx,basic,as_of=as_of)
    rec["observed_consolidations"] = observed
    rec["lookup_methods"] = ["relation", "celex_index"]
    if not observed:
        rec["error"] = "no_verified_consolidation_rows"
        return rec

    candidate = observed[0]
    rec["latest_date_candidate"] = candidate["date"]
    rec["latest_celex_candidate"] = candidate["celex"]
    rec["latest_candidate_official_url"] = candidate["official_celex_url"]

    # If either query failed, retrieval may be incomplete.
    if "relation_error" in rec or "celex_index_error" in rec:
        rec["status"] = "partial_metadata"
        rec["date_comparison"] = "latest_observed_only_incomplete_verification"
        return rec
    if len(rel) >= MAX_VERSIONS or len(idx) >= MAX_VERSIONS:
        rec["status"] = "partial_metadata"
        rec["error"] = "result_limit_reached"
        return rec

    rec["status"] = "metadata_found"
    rec["complete_consolidation_list_verified"] = False  # Graph freshness/coverage not provable here.
    snapshot = rec["snapshot_date"]
    if snapshot is None:
        rec["date_comparison"] = "original_snapshot_without_consolidation_date"
    elif candidate["date"] > snapshot:
        rec["date_comparison"] = "newer_consolidation_observed"
    elif candidate["date"] == snapshot:
        rec["date_comparison"] = "matches_latest_observed_consolidation"
    else:
        rec["date_comparison"] = "latest_observed_older_than_snapshot_review"
    return rec

# Alias preserves existing tool wiring below.
retrieve_eurlex_basic_page = retrieve_cellar_current_law

## 4. New exact provision lookup (`get_provision`) + evidence normalization

The new tool looks up legal chunks by **same regulation / exact Article, Annex, or numbered row**, rather than searching semantically. It does **not** fetch external amendment text or prove applicability. Each result is linked to its **source document CELEX** (not merely an Annex number). For a row embedded in a larger table, text must include the explicit row marker.

In [10]:
ALLOWED_REGULATIONS = frozenset(LAW_REGISTRY)
FILE_TO_LAW = {v["snapshot_file"]: k for k,v in LAW_REGISTRY.items()}

def normalized_article(value):
    m = re.fullmatch(r"(?:Article|Art\.?)[ ]*(\d+)", str(value).strip(), re.I)
    return f"Article {int(m.group(1))}" if m else None

def get_provision(regulation_id: str, provision: str, top_k: int = 5) -> list[dict]:
    """Strict local lookup: full Article, Annex heading, or explicit row marker.

    A row must be referenced as e.g. 'row 1.1.3' or 'Annex II row 14'.
    Never substitute an unrelated row just because its number occurs in prose.
    """
    if regulation_id not in ALLOWED_REGULATIONS:
        raise ValueError(f"Regulation not allowlisted: {regulation_id}")
    provision = str(provision).strip()
    if not provision or len(provision)>100:
        raise ValueError("Provision must be specific and <= 100 chars")
    top_k=max(1,min(int(top_k),8))
    article = normalized_article(provision)
    mrow = re.search(r"(?:^|\s)row\s+(\d+(?:\.\d+)*)\b", provision, re.I)
    annex = re.search(r"\bAnnex\s+(IIa|IV|III|II|I)\b", provision, re.I)
    candidates=[]
    for chunk in legal_chunks:
        if chunk.get("file") != LAW_REGISTRY[regulation_id]["snapshot_file"]:
            continue
        loc=str(chunk.get("location") or "")
        text=str(chunk.get("text") or "")
        art=str(chunk.get("article") or "")
        if article:
            matched = (art.casefold()==article.casefold())
        elif mrow:
            row=mrow.group(1)
            explicit_row = bool(re.search(r"(?:^|\bRow:)\s*"+re.escape(row)+r"(?:\s*\||\b)",text,re.I))
            location_row = bool(re.search(r"\brow\s+"+re.escape(row)+r"\b",loc,re.I))
            matched = explicit_row or location_row
            if annex:
                annex_label=annex.group(1)
                # Include table row chunks when source contains explicit numbered row,
                # but require correct Annex label for Annex-specific requests.
                matched = matched and bool(re.search(r"\bAnnex\s+"+re.escape(annex_label)+r"\b",loc+" "+text,re.I))
        elif annex:
            a=annex.group(1)
            matched = bool(re.search(r"\bAnnex\s+"+re.escape(a)+r"\b",loc,re.I))
        else:
            raise ValueError("Use an exact 'Article N', 'Annex N', or 'row X' lookup")
        if matched:
            candidates.append(chunk)
    # Entire multi-chunk article kept in chunk ID order, not relevance ranking.
    candidates.sort(key=lambda c:str(c.get("chunk_id","")))
    return [dict(c) for c in candidates[:top_k]]


def make_evidence(chunk: dict, retrieved_by: str) -> dict:
    file_name=str(chunk.get("file") or "")
    law_id=FILE_TO_LAW.get(file_name)
    if not law_id:
        raise ValueError(f"Unknown frozen file: {file_name}")
    law=LAW_REGISTRY[law_id]
    chunk_id=str(chunk.get("chunk_id") or "")
    evidence_id="E"+hashlib.sha256(chunk_id.encode()).hexdigest()[:10]
    date_part=("-"+law["snapshot_date"].replace("-", "")) if law.get("snapshot_date") else ""
    consolidated_celex=("0"+law["celex"][1:]+date_part) if date_part else law["celex"]
    return {
        "evidence_id": evidence_id, "chunk_id":chunk_id,
        "regulation_id":law_id, "source_document_celex":consolidated_celex,
        "basic_celex":law["celex"], "source_file":file_name,
        "source_version":law.get("snapshot_date") or "original_snapshot",
        "article":chunk.get("article"), "location":chunk.get("location"),
        "source_url": f'https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:{consolidated_celex}',
        "text":str(chunk.get("text") or ""),
        "retrieved_by":retrieved_by,
    }

print("get_provision() ready; allowlisted laws:", len(LAW_REGISTRY))

get_provision() ready; allowlisted laws: 5


## 5. Agent State, structured outputs and execution controls

Node functions return **partial state updates**; no mutable global Tool history is shared between different QIDs. Local provenance records are frozen by `evidence_id` (derived from `chunk_id`). Benchmark references and evaluation scores are **not in this state**.

In [11]:
class InvestigationTask(BaseModel):
    task_id: str
    question: str
    priority: Literal["high","medium","low"] = "medium"
    search_query: str = Field(min_length=5)
    expected_evidence_type: str = "relevant legal provisions"
    needs_currentness: bool = False

class PlanOutput(BaseModel):
    tasks: list[InvestigationTask] = Field(min_length=1, max_length=7)

class SuggestedGap(BaseModel):
    description: str
    query: str
    regulation_id: str | None = None
    exact_provision: str | None = None
    critical: bool = True

class LegalClaim(BaseModel):
    statement: str
    evidence_ids: list[str] = Field(default_factory=list)
    support_quote: str = ""
    status: Literal["supported", "unsupported", "conflicting", "unverified", "not_applicable"] = "unverified"
    scope_note: str = ""

class ValidationOutput(BaseModel):
    claims: list[LegalClaim] = Field(default_factory=list)
    gaps: list[SuggestedGap] = Field(default_factory=list)
    conflicts: list[str] = Field(default_factory=list)

class AuditOutput(BaseModel):
    issues: list[str] = Field(default_factory=list)
    pass_audit: bool = False
    revised_report: str = ""

class AgentState(TypedDict, total=False):
    qid: str
    question: str
    case_context: dict
    investigation_plan: list[dict]
    research_queue: list[dict]
    evidence: list[dict]
    claims: list[dict]
    gaps: list[dict]
    conflicts: list[str]
    current_law_checks: list[dict]
    search_calls: int
    initial_searches: int
    targeted_searches: int
    cellar_calls: int
    validation_rounds: int
    llm_calls: int
    usage: dict
    searched_signatures: list[str]
    trace: list[dict]
    stop_reason: str
    draft_answer: str
    final_answer: str
    audit_issues: list[str]


def new_state(question: str, qid: str = "USER", case_context: dict | None = None) -> AgentState:
    return AgentState(
        qid=qid,question=question,case_context=case_context or {},
        investigation_plan=[], research_queue=[], evidence=[],claims=[],gaps=[],
        conflicts=[], current_law_checks=[],search_calls=0,initial_searches=0,
        targeted_searches=0,cellar_calls=0,validation_rounds=0,llm_calls=0,
        usage={"input_tokens":0,"output_tokens":0}, searched_signatures=[],
        trace=[], stop_reason="",draft_answer="",final_answer="",audit_issues=[])

# Explicit run safety checks.
assert MAX_TOTAL_SEARCHES == MAX_INITIAL_SEARCHES + MAX_TARGETED_SEARCHES
assert MAX_VALIDATION_ROUNDS >= 1
assert MAX_LLM_CALLS >= 4

## 6. OpenAI LLM interface — API only when `RUN_OFFICIAL=True`

The API uses the **Responses API** pattern from NB04b (`reasoning={"effort":"medium"}`, `store=False`) to preserve model configuration. Structured responses are parsed and validated locally with Pydantic. If JSON extraction fails, the workflow **fails clearly**, rather than silently inventing a plan or validated claims.

In [12]:
CLIENT = None
SHARED_CONTEXT = nb04b_config["benchmark_context"]
SHARED_SYSTEM_PROMPT = nb04b_config["system_prompt"]
# No eval references or question-specific must-have answers enter this context.

GLOBAL_RULES = """You are assisting with EU food-law research using a frozen five-law corpus.
Never invent legal limits, section numbers, provisions, obligations or applicability.
Distinguish a regulation's own Annex from the Annex of a different amending act.
A consolidating document date / Cellar metadata is NOT proof of effective law.
Always distinguish confirmed document text, shipment facts and legal applicability.
A single chemical detection is NOT automatically a legal exceedance.
Do not claim current article-level status unless genuinely verified.
Only support a legal claim using retrieved legal text and explicit evidence IDs.
No benchmark references, evaluation rubrics or earlier model grades are accessible."""

LLM_USAGE = {"input_tokens":0,"output_tokens":0,"calls":0}

def extract_json_object(text: str) -> dict:
    s=text.strip()
    if s.startswith("```"):
        s=re.sub(r"^```(?:json)?\s*", "", s,flags=re.I)
        s=re.sub(r"\s*```$", "", s)
    try: return json.loads(s)
    except json.JSONDecodeError:
        decoder=json.JSONDecoder()
        for i,c in enumerate(s):
            if c=="{":
                try:
                    obj,offset=decoder.raw_decode(s[i:])
                    if isinstance(obj,dict):return obj
                except json.JSONDecodeError:pass
        raise ValueError("LLM output was not valid JSON object")

def llm_call(task_instructions: str, payload: dict, *, json_schema=None, max_output_tokens=MAX_OUTPUT_TOKENS):
    global CLIENT
    if not RUN_OFFICIAL:
        raise RuntimeError("LLM disabled; set RUN_OFFICIAL=True deliberately")
    if CLIENT is None:
        if not os.getenv("OPENAI_API_KEY"):
            os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY (hidden): ").strip()
        if not os.getenv("OPENAI_API_KEY"):
            raise RuntimeError("OPENAI_API_KEY missing")
        CLIENT = OpenAI()
    if LLM_USAGE["calls"]>=MAX_LLM_CALLS:
        raise RuntimeError("LLM call budget exhausted")
    response=CLIENT.responses.create(
        model=MODEL_NAME,reasoning={"effort":REASONING_EFFORT},
        instructions="\n\n".join([SHARED_CONTEXT,SHARED_SYSTEM_PROMPT,GLOBAL_RULES,task_instructions]),
        input=json.dumps(payload,ensure_ascii=False),
        max_output_tokens=max_output_tokens,store=False,
    )
    LLM_USAGE["calls"]+=1
    usage=getattr(response,"usage",None)
    LLM_USAGE["input_tokens"] += int(getattr(usage,"input_tokens",0) or 0)
    LLM_USAGE["output_tokens"] += int(getattr(usage,"output_tokens",0) or 0)
    output_text=getattr(response,"output_text",None)
    if not output_text:
        output_text="\n".join(str(getattr(part,"text","")) for block in getattr(response,"output",[]) for part in getattr(block,"content",[]))
    if not output_text.strip():
        raise RuntimeError("No model output; consider increasing max_output_tokens for reasoning")
    if json_schema is None:
        return output_text.strip()
    data=extract_json_object(output_text)
    return json_schema.model_validate(data)

def compact_sources(evidence:list[dict], max_items=24, text_chars=1350):
    # Ensure new evidence is represented; older evidence is retained in full in State.
    return [{k:e.get(k) for k in ("evidence_id","regulation_id","source_document_celex", "article","location","source_version","source_url")} |
           {"text":e["text"][:text_chars]} for e in evidence[-max_items:]]

## 7. Planner — question-specific *research intentions*, not prefilled answers

At most seven tasks are planned. Search queries may mention a regulation, but the planner must **not** treat its guesses as legal proof. The first retrieval wave is bounded to five calls.

In [13]:
def plan_investigation(state:AgentState)->dict:
    instructions="""Act ONLY as an EU regulatory INVESTIGATION PLANNER. Return a JSON object:
    {"tasks":[{"task_id":"T1","question":"...","priority":"high",
    "search_query":"...","expected_evidence_type":"...","needs_currentness":false}]}.
    Select 1 to 7 concrete research tasks tailored to the user's question.
    Cover important legal issues without reproducing answers, numeric limits or unverifiable assertions.
    Prefer high-value searches; max FIVE will be run initially. Do not use an evaluation checklist."""
    output=llm_call(instructions,{"question":state["question"],"case_context":state["case_context"]},json_schema=PlanOutput)
    tasks=[t.model_dump() for t in output.tasks]
    order={"high":0,"medium":1,"low":2}
    tasks.sort(key=lambda x:order[x["priority"]])
    queries=[{"mode":"search","query":t["search_query"],"task_id":t["task_id"],"phase":"initial","top_k":5} for t in tasks[:MAX_INITIAL_SEARCHES]]
    return {"investigation_plan":tasks,"research_queue":queries,"llm_calls":LLM_USAGE["calls"],
            "trace":state["trace"]+[{"node":"planner","planned_tasks":len(tasks),"queued":len(queries)}]}

## 8. Retriever — cached frozen corpus + exact lookup + optional Cellar

The initial wave uses **exact NB03 hybrid retrieval**; later waves use targeted `get_provision()` where the Validator specifies an exact Article or Annex row. Metadata requests (if enabled) are independently capped. Tool budgets are enforced **inside the node**, not only in a prompt.

In [14]:
def search_signature(item:dict)->str:
    return json.dumps({k:item.get(k) for k in ("mode","query","regulation_id","provision")},sort_keys=True).casefold()

def retrieve_evidence(state:AgentState)->dict:
    by_id={e["evidence_id"]:e for e in state["evidence"]}
    seen=set(state["searched_signatures"])
    trace=list(state["trace"])
    initial=state["initial_searches"]
    targeted=state["targeted_searches"]
    total=state["search_calls"]
    cellar=state["cellar_calls"]
    currentness=list(state["current_law_checks"])
    current_laws_seen={c.get("regulation") for c in currentness}
    for item in state.get("research_queue",[]):
        sig=search_signature(item)
        if sig in seen:
            trace.append({"node":"retriever","skipped_duplicate":sig})
            continue
        phase=item.get("phase","initial")
        if total>=MAX_TOTAL_SEARCHES:
            trace.append({"node":"retriever","stop":"total_search_budget"});break
        if phase=="initial" and initial>=MAX_INITIAL_SEARCHES:break
        if phase=="targeted" and targeted>=MAX_TARGETED_SEARCHES:break
        seen.add(sig)
        mode=item.get("mode","search")
        try:
            if mode=="exact":
                chunks=get_provision(item["regulation_id"],item["provision"],top_k=item.get("top_k",5))
            elif mode=="search":
                chunks=hybrid_search(item["query"],top_k=item.get("top_k",5))
            else:
                raise ValueError("Unsupported retrieval mode")
        except Exception as exc:
            chunks=[]
            trace.append({"node":"retriever","query":item,"error":str(exc)[:250]})
        total+=1
        if phase=="initial": initial+=1
        else: targeted+=1
        added=[]
        for ch in chunks:
            e=make_evidence(ch,retrieved_by=mode)
            if e["evidence_id"] not in by_id:
                by_id[e["evidence_id"]]=e
                added.append(e["evidence_id"])
        trace.append({"node":"retriever","mode":mode,"phase":phase,"query":item.get("query") or item.get("provision"),
                      "retrieved":len(chunks),"added_ids":added,"search_count":total})
    # Cellar is optional and does not silently become a provision-validity check.
    if ENABLE_CELLAR_LIVE and cellar<MAX_CELLAR_CALLS:
        for task in state.get("investigation_plan",[]):
            if not task.get("needs_currentness") or cellar>=MAX_CELLAR_CALLS:continue
            # Only known, allowlisted IDs may be queried. Planner cannot add arbitrary domains/URLs.
            planned_laws = re.findall(r"\b(?:2023/915|2019/1793|2017/625|178/2002|2019/1013)\b",task["search_query"])
            if not planned_laws:
                # Do not invent a law: infer candidates only from ACTUAL retrieved chunks.
                planned_laws = list(dict.fromkeys(e["regulation_id"] for e in by_id.values()))
            for law in planned_laws:
                if law not in current_laws_seen and cellar<MAX_CELLAR_CALLS:
                    try: record=retrieve_cellar_current_law(law)
                    except Exception as exc:record={"regulation":law,"status":"unverified","error":str(exc)}
                    currentness.append(record);current_laws_seen.add(law);cellar+=1
                    trace.append({"node":"retriever","tool":"check_current_law","regulation":law,"status":record.get("status"),
                                  "article_level_verified":False})
    return {"evidence":list(by_id.values()),"research_queue":[],"searched_signatures":sorted(seen),
            "search_calls":total,"initial_searches":initial,"targeted_searches":targeted,
            "cellar_calls":cellar,"current_law_checks":currentness,"trace":trace}

## 9. Hybrid Evidence Validator — deterministic gates + semantic review

**Deterministic checks:** evidence IDs exist, matching literal support quote occurs in retrieved text, citation points to the right **document ID**, duplicate/unsupported claims cannot be upgraded by consolidation metadata. **Semantic checks:** an LLM examines applicability, exceptions, actor/stage and cross-document amendment relations. Only the LLM **proposes** gaps; the Python gate decides whether any claim may be marked `supported`.

**Important:** a short verbatim support quote is necessary but **not sufficient** for legal applicability. The semantic check may downgrade it; the deterministic check never upgrades a claim to `supported`.

In [15]:
def normalized_ws(s:str)->str:
    return " ".join(str(s).casefold().split())

def deterministic_validate_claim(claim:dict,evidence_map:dict[str,dict])->dict:
    c=dict(claim)
    ids=list(dict.fromkeys(c.get("evidence_ids",[])))
    valid=[i for i in ids if i in evidence_map]
    missing=[i for i in ids if i not in evidence_map]
    c["evidence_ids"]=valid
    text=normalized_ws(c.get("support_quote", ""))
    quote_found=bool(len(text)>=12 and any(text in normalized_ws(evidence_map[i]["text"]) for i in valid))
    c["quote_verified"]=quote_found
    c["invalid_evidence_ids"]=missing
    if c.get("status")=="supported" and (not valid or not quote_found):
        c["status"]="unverified"
        c["scope_note"]=(c.get("scope_note","")+" | Deterministic citation/quote gate failed.").strip()
    c["current_applicability_status"]="unverified"   # Cellar metadata cannot set this to verified.
    return c

def validate_evidence(state:AgentState)->dict:
    instructions="""You are an EU legal EVIDENCE VALIDATOR, not an advocate for the draft.
    Return JSON {"claims":[{"statement":"...","evidence_ids":["E..."],
    "support_quote":"literal 12+ character excerpt from retrieved evidence",
    "status":"supported|unsupported|conflicting|unverified|not_applicable",
    "scope_note":"..."}],
    "gaps":[{"description":"...","query":"narrow search phrase","regulation_id":null,
    "exact_provision":null,"critical":true}],"conflicts":["..."]}.
    For each significant conclusion requested, require DIRECT legal text from EXACT cited evidence.
    DO NOT assume that a reference to Annex I in an amending act refers to Annex I of the law being amended;
    if the amending act text is absent from the supplied corpus, mark the alleged conflict unverified.
    Do not hallucinate requirements or numerical values. Check product, origin, stage, processing and exceptions.
    Write gaps as actionable targeted queries. Keep <= 12 claims and <= 5 gaps.
    Classify current-applicability as unverified unless actual amending text and dates were checked.
    Never use the evaluation benchmark, earlier grades, or answer checklists."""
    payload={"question":state["question"],"case_context":state["case_context"],
             "plan":state["investigation_plan"],"evidence":compact_sources(state["evidence"]),
             "current_law_metadata": [{k:x.get(k) for k in ("regulation","status","latest_date_candidate","article_level_changes_verified","application_date_verified")} for x in state["current_law_checks"]]}
    output=llm_call(instructions,payload,json_schema=ValidationOutput,max_output_tokens=3800)
    known={e["evidence_id"]:e for e in state["evidence"]}
    claims=[deterministic_validate_claim(c.model_dump(),known) for c in output.claims[:12]]
    gaps=[g.model_dump() for g in output.gaps[:5]]
    # Claim-side missing proof is also a gap even if the model missed it.
    for c in claims:
        if c["status"] in ("unsupported","conflicting","unverified") and len(gaps)<7:
            gaps.append({"description":"Missing support: "+c["statement"][:160],
                         "query":c["statement"][:230],"regulation_id":None,"exact_provision":None,"critical":True})
    rounds=state["validation_rounds"]+1
    return {"claims":claims,"gaps":gaps,"conflicts":output.conflicts,
            "validation_rounds":rounds,"llm_calls":LLM_USAGE["calls"],
            "trace":state["trace"]+[{"node":"validator","round":rounds,"claims":len(claims),
                                  "supported":sum(c["status"]=="supported" for c in claims),"gaps":len(gaps),
                                  "conflicts":len(output.conflicts)}]}

## 10. Stopping rules + targeted re-search queue

The **retry budget is enforced in Python**. A second validation pass cannot trigger infinite loops. Re-search is used only for a specific unresolved **material** issue, with explicit priority and deduplication. If no improvement is possible, output a qualified report instead of hallucinating missing law.

In [16]:
def decide_next_step(state:AgentState)->Literal["targeted_research","report","report_with_gaps"]:
    critical_gaps=[g for g in state.get("gaps",[]) if g.get("critical",True)]
    unsupported=[c for c in state.get("claims",[]) if c.get("status") not in ("supported","not_applicable")]
    if not critical_gaps and not state.get("conflicts") and not unsupported:
        return "report"
    if state["validation_rounds"] >= MAX_VALIDATION_ROUNDS:
        return "report_with_gaps"
    if state["targeted_searches"] >= MAX_TARGETED_SEARCHES or state["search_calls"]>=MAX_TOTAL_SEARCHES:
        return "report_with_gaps"
    # Avoid a loop if no new, material, distinct search is available.
    pending=build_targeted_queue(state)
    return "targeted_research" if pending else "report_with_gaps"

def build_targeted_queue(state:AgentState)->list[dict]:
    seen=set(state.get("searched_signatures",[]))
    slots=max(0,min(MAX_TARGETED_SEARCHES-state["targeted_searches"],MAX_TOTAL_SEARCHES-state["search_calls"]))
    out=[]
    for gap in [g for g in state.get("gaps",[]) if g.get("critical",True)]:
        law=gap.get("regulation_id")
        provision=gap.get("exact_provision")
        exact=bool(law in ALLOWED_REGULATIONS and provision)
        item={"phase":"targeted","mode":"exact" if exact else "search",
              "query":gap.get("query") or gap.get("description",""),"top_k":5}
        if exact:item.update({"regulation_id":law,"provision":provision})
        if len(item["query"].strip())<5:continue
        sig=search_signature(item)
        if sig in seen:continue
        seen.add(sig);out.append(item)
        if len(out)>=slots:break
    return out

def plan_research_gap(state:AgentState)->dict:
    q=build_targeted_queue(state)
    return {"research_queue":q,"trace":state["trace"]+[{"node":"gap_planner","targeted_queries":q}]}

## 11. Reporter + Final Claim Audit (no novel unsupported requirements)

The Reporter receives **only verified claims as potentially confirmable**. Missing facts, unsupported claims, shipment-specific uncertainty and snapshot metadata must be separated. The final Audit checks citation IDs, unsupported statements and legal-vs-precautionary language. If audit fails, a **deterministic fallback** produces a narrower, source-linked memo (never the unverified draft).

In [17]:
def safe_fallback_report(state:AgentState, issues:list[str]|None=None)->str:
    accepted=[c for c in state.get("claims",[]) if c.get("status")=="supported" and c.get("quote_verified")]
    parts=["## Evidence-supported findings (frozen snapshot)"]
    if accepted:
        for c in accepted:
            cites=" ".join("["+i+"]" for i in c["evidence_ids"])
            parts.append("- "+c["statement"].strip()+" "+cites)
    else:parts.append("- No sufficiently verified legal conclusion available from the retrieved material.")
    parts.extend(["","## Information requiring verification"])
    for g in state.get("gaps",[])[:8]:parts.append("- "+str(g.get("description","")))
    parts.append("- The applicable law and shipment facts, including amendments and effective dates, were NOT fully verified.")
    parts.append("- A detection alone does not establish a shipment-specific legal exceedance or clearance decision.")
    parts.append("\n## Provenance and limits\n- Snapshot evidence only; current legal applicability unverified.")
    if issues:parts.append("- Audit issues: "+"; ".join(issues[:5]))
    return "\n".join(parts)

def draft_report(state:AgentState)->dict:
    supported=[c for c in state["claims"] if c["status"]=="supported" and c.get("quote_verified")]
    instruction="""Write a concise EU food business operator regulatory research memo.
    Include sections 1. Short assessment, 2. Evidence-supported law, 3. Shipment facts and currentness to verify, 4. Practical next checks.
    Every material legal statement MUST cite only its provided [EvidenceID]. Do not invent law, annex, limit or document.
    Treat snapshot document evidence as historically verified only; never assert current applicability from Cellar metadata.
    Clearly distinguish a prudent precautionary recommendation from legally mandatory detention, refusal or recall.
    If evidence is insufficient, say so. Do not invent a 'confirmed' legal requirement.
    Do not cite evidence IDs not in the supplied list."""
    # If LLM calls cannot be safely made, let the next step do a conservative fallback.
    if not supported:
        draft=safe_fallback_report(state)
    else:
        draft=llm_call(instruction,{"question":state["question"],"supported_claims":supported,
                     "gaps":state["gaps"],"conflicts":state["conflicts"],
                     "sources":compact_sources(state["evidence"],max_items=16,text_chars=900)},max_output_tokens=3000)
    return {"draft_answer":draft,"llm_calls":LLM_USAGE["calls"],
            "trace":state["trace"]+[{"node":"reporter","length":len(draft)}]}

def audit_report(state:AgentState)->dict:
    known={e["evidence_id"] for e in state["evidence"]}
    cited=set(re.findall(r"\[(E[0-9a-f]{10})\]",state["draft_answer"]))
    invalid=sorted(cited-known)
    issues=["Unknown evidence IDs: "+", ".join(invalid)] if invalid else []
    if state["claims"] and not cited and any(c["status"]=="supported" for c in state["claims"]):
        issues.append("Draft lacks evidence-ID citations")
    # Semantic check is a second independent model invocation with constrained data.
    if state["claims"] and not issues:
        audit_instructions="""Audit the draft for unsupported legal claims and misleading certainty.
        Return JSON {"issues":["..."], "pass_audit":true|false, "revised_report":"..."}.
        All legal conclusions must be supported by provided verified claims and correct evidence IDs.
        Never infer article applicability from Cellar metadata. Distinguish precautionary recommendations
        from legally compelled holds/refusals. If any material unsupported claim exists, pass_audit=false.
        If revision is needed, write a conservative revised_report without unsupported legal claims.
        Keep verified source IDs unchanged."""
        try:
            audit=llm_call(audit_instructions,{"draft":state["draft_answer"],
                         "supported_claims":[c for c in state["claims"] if c["status"]=="supported"],
                         "gaps":state["gaps"]},json_schema=AuditOutput,max_output_tokens=2500)
            issues.extend(audit.issues)
            # Audited revision is still restricted to known citation IDs.
            candidate=audit.revised_report or state["draft_answer"]
            candidate_invalid=set(re.findall(r"\[(E[0-9a-f]{10})\]",candidate))-known
            if audit.pass_audit and not candidate_invalid:
                final=candidate
            else:
                final=safe_fallback_report(state,issues+["Semantic audit did not pass"])
        except Exception as exc:
            issues.append("Semantic audit error: "+type(exc).__name__+": "+str(exc)[:150])
            final=safe_fallback_report(state,issues)
    elif issues:
        final=safe_fallback_report(state,issues)
    else:
        final=state["draft_answer"]
    return {"final_answer":final,"audit_issues":issues,"llm_calls":LLM_USAGE["calls"],
            "usage":{"input_tokens":LLM_USAGE["input_tokens"],"output_tokens":LLM_USAGE["output_tokens"]},
            "trace":state["trace"]+[{"node":"auditor","issues":issues,"used_fallback":final!=state["draft_answer"]}]}

## 12. Compile the LangGraph Investigation Agent

A conditional edge enforces at most two validation passes. LangGraph manages state transitions, while the Retriever and Stopping Rules enforce the actual budgets.

In [18]:
workflow=StateGraph(AgentState)
workflow.add_node("planner",plan_investigation)
workflow.add_node("retriever",retrieve_evidence)
workflow.add_node("validator",validate_evidence)
workflow.add_node("gap_planner",plan_research_gap)
workflow.add_node("reporter",draft_report)
workflow.add_node("auditor",audit_report)
workflow.add_edge(START,"planner")
workflow.add_edge("planner","retriever")
workflow.add_edge("retriever","validator")
workflow.add_conditional_edges("validator",decide_next_step,{
    "targeted_research":"gap_planner",
    "report":"reporter",
    "report_with_gaps":"reporter",
})
workflow.add_edge("gap_planner","retriever")
workflow.add_edge("reporter","auditor")
workflow.add_edge("auditor",END)
agent_graph=workflow.compile()
print("Build04 graph compiled: planner → retriever → validator ↔ gap_planner → reporter → auditor")

Build04 graph compiled: planner → retriever → validator ↔ gap_planner → reporter → auditor


## 13. Offline tests (no API / network / paid calls)

Test: correct known Article and row lookup, document-identity isolation, support-quote gate, missing-evidence downgrade, no infinite re-search, budget stopping, and safe report fallback. These tests do **not** assess legal correctness of the actual generated report.

In [19]:
if not RUN_OFFICIAL:

    # Exact Article test on the actual NB08 frozen chunks.
    a=get_provision("2019/1793","Article 11")
    assert a and all(x.get("article")=="Article 11" for x in a)
    figrow=get_provision("2023/915","row 1.1.3")
    assert figrow and any(re.search(r"Row:\s*1\.1\.3\b",x.get("text","")) for x in figrow)
    annex_ii_figs=get_provision("2019/1793","Annex II row 14")
    assert annex_ii_figs and all("Annex II" in str(x.get("location","")) for x in annex_ii_figs)
    assert not get_provision("2023/915","Article 9999")
    try:
        get_provision("2025/1441","Annex I")
        raise AssertionError("Non-corpus amending act incorrectly allowed")
    except ValueError:pass

    sample=make_evidence(figrow[0],"exact")
    needle=sample["text"][20:80]
    valid={"statement":"Example claim about a source text","evidence_ids":[sample["evidence_id"]],
           "support_quote":needle,"status":"supported","scope_note":""}
    audit=deterministic_validate_claim(valid,{sample["evidence_id"]:sample})
    assert audit["status"]=="supported" and audit["quote_verified"]
    invalid=dict(valid,support_quote="A claim that the evidence never states")
    assert deterministic_validate_claim(invalid,{sample["evidence_id"]:sample})["status"]=="unverified"
    assert deterministic_validate_claim(dict(valid,evidence_ids=["Enotfound"]),{sample["evidence_id"]:sample})["status"]=="unverified"

    s=new_state("Synthetic question for stopping tests")
    s.update({"validation_rounds":MAX_VALIDATION_ROUNDS,"gaps":[{"description":"missing","query":"missing legal source","critical":True}]})
    assert decide_next_step(s)=="report_with_gaps"
    s2=new_state("Synthetic question for stopping tests")
    s2.update({"gaps":[{"description":"missing","query":"missing legal source","critical":True}]})
    assert decide_next_step(s2)=="targeted_research"
    s3=new_state("Synthetic question for stopping tests")
    s3.update({"gaps":[],"claims":[],"conflicts":[]})
    assert decide_next_step(s3)=="report"
    print("PASS: exact provisions, source identity, quote gate, no-evidence downgrade and stop rules")
else:
    print("SKIP offline self-tests during paid run (already tested with RUN_OFFICIAL=False).")

PASS: exact provisions, source identity, quote gate, no-evidence downgrade and stop rules


In [20]:
RUN_OFFICIAL = True
SELECT_QIDS = ["Q4"]
ENABLE_CELLAR_LIVE = False

print("Official run enabled:", RUN_OFFICIAL)
print("Selected questions:", SELECT_QIDS)

Official run enabled: True
Selected questions: ['Q4']


## 14. Official run — OFF by default; start with one QID

**Requires OpenAI API usage.** Execute the notebook offline tests first, then set `RUN_OFFICIAL=True` **in the setup cell**, rerun that setup and these cells. The first recommended QID is `Q4`; for a full run use `SELECT_QIDS=list(questions)` after reviewing Q4. Optional live Cellar can be enabled separately, but is not mandatory for a frozen-corpus test. Runs save one `Q#.json` per question and **never overwrite a prior official file**.

If notebook is running in an isolated container, it cannot make paid OpenAI calls without the user's own API key and network. The generated notebook has **not** made any paid calls yet.

In [21]:
# Research budget and manifest gate, required before any paid question runs.
assert PARITY_OK and matched==50, "Retrieval parity must be 50/50"
assert all(qid in questions for qid in SELECT_QIDS)
RUN_DIR.mkdir(parents=True,exist_ok=True)
RUN_SETTINGS={
    "experiment_id":EXPERIMENT_ID,"benchmark_sha256":benchmark_hash,
    "model":MODEL_NAME,"reasoning_effort":REASONING_EFFORT,
    "chunks_sha256":hashlib.sha256(FILES["chunks"].read_bytes()).hexdigest(),
    "retrieval_config_sha256":hashlib.sha256(FILES["retrieval_config"].read_bytes()).hexdigest(),
    "initial_max":MAX_INITIAL_SEARCHES,"targeted_max":MAX_TARGETED_SEARCHES,
    "validation_max":MAX_VALIDATION_ROUNDS,"max_llm_calls":MAX_LLM_CALLS,
    "enable_cellar_live":ENABLE_CELLAR_LIVE,
    "runtime_version":"nb09_build04_v1",
}
manifest_path=RUN_DIR/"manifest.json"
if RUN_OFFICIAL:
    if manifest_path.exists():
        previous=json.loads(manifest_path.read_text(encoding="utf-8"))
        assert previous==RUN_SETTINGS,"Experiment configuration changed. Use a new EXPERIMENT_ID / directory."
    else:
        with manifest_path.open("x",encoding="utf-8") as f:
            json.dump(RUN_SETTINGS,f,indent=2,ensure_ascii=False)
else:
    print("DRY RUN: manifest not written; you may adjust settings before the first official run.")
print("Parity",matched,"/50 | output",RUN_DIR,"| paid runs",RUN_OFFICIAL,"| selected",SELECT_QIDS)

Parity 50 /50 | output data\build04_nb09_runs | paid runs True | selected ['Q4']


In [22]:
def execute_one_qid(qid:str)->dict|None:
    assert PARITY_OK
    assert qid in questions
    outpath=RUN_DIR/f"{qid}.json"
    if outpath.exists():
        print("SKIP: previous result is preserved:",outpath)
        return None
    start=datetime.now(timezone.utc)
    t0=time.perf_counter()
    LLM_USAGE.update({"input_tokens":0,"output_tokens":0,"calls":0})
    state=new_state(questions[qid],qid,case_context={"benchmark_case":"Türkiye-origin dried figs; aflatoxin B1 detected; measured concentration and consignment status not provided"})
    try:
        answer=agent_graph.invoke(state,config={"recursion_limit":20})
        assert answer.get("final_answer"),"Empty final report"
        record={
            "experiment_id":EXPERIMENT_ID,"build":"Build04","qid":qid,
            "question":questions[qid],"benchmark_id":benchmark["benchmark_id"],
            "benchmark_sha256":benchmark_hash,"settings":RUN_SETTINGS,
            "model":MODEL_NAME,"reasoning_effort":REASONING_EFFORT,
            "plan":answer.get("investigation_plan",[]),
            "evidence":answer.get("evidence",[]),"claims":answer.get("claims",[]),
            "gaps":answer.get("gaps",[]),"conflicts":answer.get("conflicts",[]),
            "current_law_calls":answer.get("current_law_checks",[]),
            "trace":answer.get("trace",[]),"final_output":answer["final_answer"],
            "search_calls":answer.get("search_calls",0),
            "initial_searches":answer.get("initial_searches",0),
            "targeted_searches":answer.get("targeted_searches",0),
            "validation_rounds":answer.get("validation_rounds",0),
            "llm_calls":LLM_USAGE["calls"],"usage":answer.get("usage",{}),
            "audit_issues":answer.get("audit_issues",[]),
            "started_utc":start.isoformat(),"completed_utc":datetime.now(timezone.utc).isoformat(),
            "latency_seconds":round(time.perf_counter()-t0,2),
            "status":"completed_pending_human_review",
        }
        with outpath.open("x",encoding="utf-8") as f:
            json.dump(record,f,ensure_ascii=False,indent=2)
        print("SAVED",qid,"| searches",record["search_calls"],
              "| rounds",record["validation_rounds"],"| LLM calls",record["llm_calls"],
              "| tokens",record["usage"])
        return record
    except Exception as exc:
        err={"qid":qid,"error_type":type(exc).__name__,"error_message":str(exc),
             "failed_utc":datetime.now(timezone.utc).isoformat(),"llm_usage":dict(LLM_USAGE)}
        (RUN_DIR/f"{qid}_error.json").write_text(json.dumps(err,ensure_ascii=False,indent=2),encoding="utf-8")
        print("FAILED",qid,err["error_type"],err["error_message"][:350])
        raise

if RUN_OFFICIAL:
    for qid in SELECT_QIDS:
        execute_one_qid(qid)
else:
    print("Official run disabled. No OpenAI requests made.")

SAVED Q4 | searches 7 | rounds 2 | LLM calls 5 | tokens {'input_tokens': 13673, 'output_tokens': 6130}


## 15. Read-only comparison & human evaluation worksheet

Build01/02 answer text is read from the frozen NB04b CSV. Build03-B NB08 saved outputs are **optional external records**, not required for executing Build04. **Correct/Grounded/Complete remain human-assigned** against the frozen Verified Reference after the run; never expose those grades to the Agent.

The filename `Q#.json` under `data/build03_nb08_B_official_runs/` is the NB08 format. If those records are stored somewhere else, adjust `BUILD03_DIR` manually. No automatic grades are generated.

In [ ]:
BUILD03_DIR=DATA_DIR/"build03_nb08_B_official_runs"
rows=[]
for qid in questions:
    p=RUN_DIR/f"{qid}.json"
    obj=json.loads(p.read_text(encoding="utf-8")) if p.exists() else None
    base03=BUILD03_DIR/f"{qid}.json"
    old=json.loads(base03.read_text(encoding="utf-8")) if base03.exists() else None
    rows.append({
        "qid":qid,
        "Build02_frozen_answer":comparison_df.loc[comparison_df.qid==qid,"rag_answer"].iloc[0][:250],
        "Build03_answer_available":bool(old),
        "Build04_status":obj["status"] if obj else "NOT_RUN",
        "Build04_searches":obj.get("search_calls") if obj else None,
        "Build04_targeted":obj.get("targeted_searches") if obj else None,
        "Build04_supported_claims":sum(c.get("status")=="supported" for c in obj["claims"]) if obj else None,
        "Build04_audit_issues":len(obj.get("audit_issues",[])) if obj else None,
    })
summary_df=pd.DataFrame(rows)
display(summary_df)
# Blank worksheet: grading is done by a human against fixed Notion reference.
EVAL_PATH=RUN_DIR/"build04_q1_q10_human_evaluation.csv"
if not EVAL_PATH.exists():
    pd.DataFrame([{
        "qid":qid,"correct":"","grounded":"","complete":"",
        "verified_reference_checked":"","critical_legal_error":"",
        "failure_category":"","notes":""
    } for qid in questions]).to_csv(EVAL_PATH,index=False,encoding="utf-8-sig")
print("Evaluation worksheet:",EVAL_PATH)

## 16. Interpretation and next experiment

- **Supported claim** = a structured LLM judgment with at least one existing evidence ID and an exact support quote found in that retrieved text; it does **not** automatically mean shipment-specific applicability or truly current law.
- **Grounded/Correct/Complete labels** must still be assigned by domain review. A high supported-claim count is not proof of legal accuracy.
- Q4 requires cross-document identity awareness: the five-law frozen corpus may not include a particular amending act. **Correct response when amendment text is unavailable is to flag that relationship as unverified**, not to assert an Annex migration or fictitious conflict.
- Source-level metadata cannot prove current applicability. A later enhancement can fetch and compare the *operative text* of specific official amendments and their dates, with source verification and regression tests.
- `MAX_TOTAL_SEARCHES=8` is an experiment budget, not a guarantee of complete retrieval. Increase it only in a **new named experiment** after examining tool traces.
- Q1–Q10 are development-visible benchmark questions. Before claiming generalization, test additional previously unseen product-origin-hazard cases.

**Next:** Run Q4 first, inspect Planner tasks, `get_provision` calls, claim-status decisions and `audit_issues`, then Q9. Evaluate against the separately maintained frozen verified reference.